For the past year, I've been relearning Python so to simultaneously further and test my knowledge I decided to investigate major city housing construction over the past year using Census data. 

With data from the Census Building Permit Survey, a survey widely-used to determine where the most building is occurring, I examined the top 10 most populated combined statistical areas (metros) where more than 105 million Americans live to learn which major metros were building the most and least amount of homes.

You can read my full analysis here: https://www.tajairi.com/Articles/Metro_Housing_Analysis.html

In [ ]:
import pandas as pd 
from pathlib import Path
import plotly.express as px
import re

internal_directory = Path.cwd()
py_directory = Path.cwd().parent
data_directory = Path(internal_directory/'housing_data')


pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 500)

/home/tajairineuson/Desktop/Code/Python/housing_examiner


In [ ]:
#Loading the Excel Sheets for each year
#Current Era
df_2019 = pd.read_excel(data_directory/'housing_2019.xls',skiprows=5,header=0)
df_2020 = pd.read_excel(data_directory/'housing_2020.xls',skiprows=5,header=0)
df_2021 = pd.read_excel(data_directory/'housing_2021.xls',skiprows=5,header=0)
df_2022 = pd.read_excel(data_directory/'housing_2022.xls',skiprows=5,header=0)
df_2023 = pd.read_excel(data_directory/'housing_2023.xls',skiprows=5,header=0)
df_2024 = pd.read_excel(data_directory/'housing_2024.xls',skiprows=5,header=0)
df_2025 = pd.read_excel(data_directory/'housing_2025prelim.xls',skiprows=7,header=0)

columns_headers = df_2019.columns.to_list()

#Population Data
pop_2024 = pd.read_csv(data_directory/'population_estimate_2020-2024.csv',
                       encoding='UTF-8')

#Filtering the csv to only include Most Populous Metros
pop_2024 = pop_2024.query("LSAD == 'Combined Statistical Area'")

pop_2024['CSA'] = pop_2024['CSA'].astype(int)

#Removes columns which have 50 or more NaN entries
pop_2024 = pop_2024.dropna(axis='columns',thresh=50)

#Top 10 Most populated Metros major codes 
metro_areas_CBSA = [12060,14460,16980,19100,26420,31080,
                    33100,35620,37980,47900]

A function that polishes and filters the dataframes to only include data from the top-10 metros since that is where around 105 million Americans live and a substantial amount of US economic activity occurs.

In [18]:
def extract_top_metros(df: pd.DataFrame,roll_year: int):
    '''Extracts the top 10 metros from the list and changes their name to make them more readable. Also create a new column with year'''
    # Drops the empty rows
    df = df.dropna(axis=0)

    #Remove the white space from the name column 
    df['Name'] = df['Name'].str.strip()

    #Converts the df to an INT
    df['CBSA'] = df['CBSA'].astype(int)
    df['CSA'] = df['CSA'].astype(int)

    #Filters dataframe to only Top 10 Metros
    df = df[df['CBSA'].isin(metro_areas_CBSA)]

    #Checks if the number of cities in metro areas CBSA matches those in the df 
    if len(df) != len(metro_areas_CBSA):
        print('You are missing some metros')
        obtained_cities = df['CBSA'].to_list()
        
        for metro in metro_areas_CBSA:
            if metro not in obtained_cities:
                print('You are missing ',metro)
        
    #Replaces everything after the - in the name column
    df['Name'] = df['Name'].str.replace('-.*',' Metro',regex=True)

    #Sets every entry = to the roll year
    df['Roll_Year'] = roll_year

    df.loc[df['Name'] == 'Washington Metro','Name'] = 'DC Metro'

    df.loc[df['Name'] == 'New York Metro','Name'] = 'New York City Metro'

    return df


I loaded the most recent years from 2019 -> 2025. 
This time-period was an anomalous period for housing development due to the Covid-19 pandemic causing a bust then boom housing building cycle.


In [ ]:
current_housing_dataframes = [df_2019,df_2020,df_2021,
           df_2022,df_2023,df_2024,
           df_2025]


current_housing_dataframes = list (
        map (
        lambda data: extract_top_metros(data[0],data[1]),zip(current_housing_dataframes,range(2019,2026))
        )
    )

## Historic Housing Data

I pulled data from 2014-2018 to understand how housing development looked during a more stable period. 



In [20]:
historic_housing_files = [  data_directory/'housing_2014.txt',
                            data_directory/'housing_2015.txt',
                            data_directory/'housing_2016.txt',
                            data_directory/'housing_2017.txt',
                            data_directory/'housing_2018.txt']

However, the 2014-2018 Census files were not in the same excel format as the later years so I had to use Regex to extract text from the files and load it into the same structure that matched the 2019 and beyond dataframes

In [21]:
#Regex Pattern for extracting housing data 
text_pattern = r'(\d+)\s(\d+)\s([\s\S]*?)[ ]{2,}(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)'

def convert_historic_txt_file(file_name: str) -> list:
    '''Converts the text file into a dataframe readable format'''

    converting_col = ['CSA', 'CBSA', 'Total', '1 Unit', 
                      '2 Units', '3 and 4 Units', '5 Units or More', 
                      'Num of Structures With 5 Units or More']

    with open(file_name,'r') as housing_file:
        housing_content = housing_file.read()
        patterns = re.findall(text_pattern,housing_content,re.MULTILINE)

    df = pd.DataFrame(patterns,
                      columns=columns_headers)
    
    df.dropna(inplace=True)

    #Converts specific columns from String to int
    df[converting_col] = df[converting_col].astype(int)
    
    #Gets rid of the new line and space that are in some of the metro names
    df['Name'] = df['Name'].str.replace('\n ','')

    return df

Once extracted, I followed the same approach as before, transformed the data, extracted the top 10 metros, converting them into analyzable dataframes. 

In [22]:
cleaned_historic_dataframes = list(map(convert_historic_txt_file,historic_housing_files))

df_2014,df_2015,df_2016,df_2017,df_2018 = list (
                                    map(
                                        lambda f: extract_top_metros(f[0],f[1]),zip(cleaned_historic_dataframes,range(2014,2019)))
)

historic_housing_dataframes = [df_2014,df_2015,df_2016,df_2017,df_2018]


I split the time periods up to match the census's methodology (current era housing being excel sheets and historic housing being txt files). In the future I plan to break them up across more useful eras ie: pre-pandemic,pandemic, and post-pandemic

In [23]:
all_time_housing = pd.concat(objs=[*historic_housing_dataframes,
                            *current_housing_dataframes],
                            ignore_index=True)

historic_housing = pd.concat(objs=[*historic_housing_dataframes],ignore_index=True)

current_housing = pd.concat(objs=[*current_housing_dataframes],ignore_index=True)

years = all_time_housing['Roll_Year'].to_list()


In [ ]:
pop_2010_2020 = pd.read_excel(data_directory/'pop_data_2010-2020.xlsx',skiprows=3,header=0,nrows=178)
city_names = all_time_housing.Name.unique().tolist()

#Creates a list with just the city name :Los Angeles Metro -> Los Angeles
city_names = [c.split(' Metro')[0] for c in city_names]

pop_2010_2020.dropna(inplace=True)

list_of_metros = []
for c in city_names:
    pattern=f'^{c}'

    if pattern == '^New York City':
        pattern = '^New York'

    list_of_metros.append(pattern)

#Filtering it to only include the top 10 Metros 
pop_2010_2020 = pop_2010_2020[pop_2010_2020['Geographic Area'].str.contains('|'.join(list_of_metros))]

no_use_columns = [2010,2011,2012,2013,'April 1, 2010 Estimates Base','April 1, 2020 Census']

pop_2010_2020.drop(columns=no_use_columns,inplace=True)

#Replaces everything after the dash in the Name with Metro 
pop_2010_2020['Geographic Area'] = pop_2010_2020['Geographic Area'].replace(regex='-.+',value=' Metro')

pop_2010_2020 = pop_2010_2020.rename(columns={'Geographic Area':'Name'})

pop_2010_2020 = pop_2010_2020.convert_dtypes(convert_floating=False)

#Grabs the CSA from housing df then creates a new column in the pop df with these numbers
# If the order changes for housing this will break. Future refactor create a better fix 
top_ten_metros_csas = all_time_housing['CSA'][:10].to_list()
pop_2010_2020.loc[:,'CSA'] = top_ten_metros_csas 


Index([             'Geographic Area', 'April 1, 2010 Estimates Base',
                                 2010,                           2011,
                                 2012,                           2013,
                                 2014,                           2015,
                                 2016,                           2017,
                                 2018,                           2019,
               'April 1, 2020 Census'],
      dtype='object')
Index(['Geographic Area', 2014, 2015, 2016, 2017, 2018, 2019], dtype='object')


Wrote a function to combine the housing and population dataframes for all years. 

In [ ]:
def merge_pop_housing_data(df: pd.DataFrame,year:int) -> pd.DataFrame:
    """Combines housing dataframe with the corresponding years population data"""
    
    if 2020 <= year < 2025:
        df = pd.merge(df,pop_2024[['CSA',f'POPESTIMATE{year}']],'inner',on='CSA')

        df.rename(columns={f'POPESTIMATE{year}':'Pop_Estimate'},inplace=True)

        df.loc[:,'Housing_Units_Per_Capita'] = round(df['Total'] / df['Pop_Estimate'],4)
        df.loc[:,'Housing_Units_Per_100k'] = df['Housing_Units_Per_Capita'] * 100_000

    elif 2010 <= year < 2020:
        df = pd.merge(df,pop_2010_2020[['CSA',year]],'inner',on='CSA')
        
        df['Roll_Year'] = year

        df.loc[:,'Housing_Units_Per_Capita'] = round(df['Total'] / df[year],4)
        df.loc[:,'Housing_Units_Per_100k'] = df['Housing_Units_Per_Capita'] * 100_000

        df.rename(columns={year:'Pop_Estimate'},inplace=True)

    else:
         raise ValueError("Sorry the year you entered I do not yet have the data for. " \
                        "Please only enter years between 2014 and 2024")

    return df

In [69]:
#Didn't want to use a map here for readability, but definitely an area for a future refactor 
housing_and_pop2014 = merge_pop_housing_data(df_2014,2014)
housing_and_pop2015 = merge_pop_housing_data(df_2015,2015)
housing_and_pop2016 = merge_pop_housing_data(df_2016,2016)
housing_and_pop2017 = merge_pop_housing_data(df_2017,2017)
housing_and_pop2018 = merge_pop_housing_data(df_2018,2018)
housing_and_pop2019 = merge_pop_housing_data(df_2019,2019)
housing_and_pop2020 = merge_pop_housing_data(df_2020,2020)
housing_and_pop2021 = merge_pop_housing_data(df_2021,2021)
housing_and_pop2022 = merge_pop_housing_data(df_2022,2022)
housing_and_pop2023 = merge_pop_housing_data(df_2023,2023)
housing_and_pop2024 = merge_pop_housing_data(df_2024,2024)


historic_housing_and_pop_list = [housing_and_pop2014,housing_and_pop2015,
                        housing_and_pop2016,housing_and_pop2017,
                        housing_and_pop2018]

housing_and_pop_list = [housing_and_pop2019,housing_and_pop2020,housing_and_pop2021,
                        housing_and_pop2022,housing_and_pop2023,
                        housing_and_pop2024]

housing_and_pop_list = list (
                        map(
                           lambda z:extract_top_metros(z[0],z[1]),zip(housing_and_pop_list,range(2019,2025)) 
                        )
)

all_years_housing_and_pop = pd.concat([*housing_and_pop_list,*historic_housing_and_pop_list],ignore_index=True)



# The Results

### Total housing Permitted (2014-2025)

In [70]:
UNIT_COLORS = ['#7B5EA7', '#70C0BA', '#FFC000', '#ED7D31']

TITLE_STYLE = dict(
    font=dict(size=22, color='#222222', family='Arial'),
    xanchor='center',
    x=0.5
)

SHARED_TITLE = 'Most Populous Metros'


total_housing = all_time_housing.groupby(by='Name',as_index=False).sum().sort_values(by='Total',ascending=False)

fig = px.bar(total_housing,
             text_auto=True,
             x='Total',
             y='Name',
             orientation='h',
             color_discrete_sequence=UNIT_COLORS,
             labels={'Name':'Metro Area','Total':'Units Permitted'})

fig.update_layout(title={**TITLE_STYLE,'text': '<b>Total Housing Permitted (2014–2025) </b>'},
                font=dict(family='Arial', size=13),
                legend=dict(title=None),
                plot_bgcolor='white',
                xaxis=dict(gridcolor='#E5E5E5', gridwidth=1),
                yaxis=dict(linecolor='#CCCCCC', categoryorder='total ascending'))

fig.update_traces(insidetextfont=dict(color='white'), outsidetextfont=dict(color='#333333'))

Texas cities permitted more housing than the bottom 6 Metros

In [71]:
texas_cities_total = total_housing[total_housing['Name'].isin(['Dallas Metro','Houston Metro'])].Total.sum()
bottom_6_total = total_housing.tail(6).Total.sum()

print(f'Texas cities permitted {texas_cities_total:,} units')

print(f'Bottom 6 Metros permitted {bottom_6_total:,} units')


Texas cities permitted 1,509,798.0 units
Bottom 6 Metros permitted 1,465,355.0 units


### Total housing by Unit type (2014-2025)

In [72]:
fig_horizontal = px.bar(total_housing,
                        text_auto=True,
             labels={'Name': 'Metro Area', 'value': 'Units Permitted'},
             y='Name',
             x=['1 Unit', '2 Units', '3 and 4 Units', '5 Units or More'],
             orientation='h',
             color_discrete_sequence=UNIT_COLORS)

fig_horizontal.update_layout(
    title={**TITLE_STYLE, 'text': '<b>Housing Permitted by Unit Type (2014–2025)</b>'},
    font=dict(family='Arial', size=13),
    legend=dict(title=None),
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#E5E5E5', gridwidth=1),
    yaxis=dict(linecolor='#CCCCCC', categoryorder='total ascending')
)

fig_horizontal.update_traces(insidetextfont=dict(color='white'), outsidetextfont=dict(color='#333333'))


fig_horizontal.show()


### Metro's Percentage of Housing Type (2014-2025)

In [73]:
all_time_housing_type = all_time_housing.groupby(by='Name',as_index=False).sum()

all_time_housing_type = all_time_housing.melt(id_vars=['Name'],
            value_vars=['1 Unit','2 Units','3 and 4 Units','5 Units or More'],
            var_name='Unit_Type',
            value_name='Units_Amount')

fig = px.pie(all_time_housing_type,
            names='Unit_Type',
            values='Units_Amount',
            facet_col='Name',
            facet_col_wrap=3,
            width=1300,
            height=1500)

fig.update_layout(
    title={**TITLE_STYLE, 'text': "<b>Metro Housing Inventory by Unit Type (2014–2025)</b>"},
    font=dict(family='Arial', size=13),
    legend=dict(title=None),
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()


### Apartment Buildings Permitted Per Metro 2014-2025

In [74]:
fig = px.scatter(all_time_housing,
                 x='Roll_Year',
                 y='Num of Structures With 5 Units or More',
                 trendline_color_override="black",
                 color='Name',
                 color_discrete_sequence=px.colors.qualitative.Pastel,
                 facet_col='Name',
                 facet_col_wrap=3,
                 labels={'Roll_Year': 'Year', 'Num of Structures With 5 Units or More': 'Structures Permitted', 'Name': 'Metro Area'},
                 width=1300,
                 height=1500)

fig.update_layout(
    title={**TITLE_STYLE, 'text': f'<b>Apartment Buildings Permitted: {SHARED_TITLE}</b>'},
    font=dict(family='Arial', size=13),
    legend=dict(title=None),
    plot_bgcolor='white',
    
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.update_traces(marker=dict(size=20, symbol="square-cross", line=dict(width=2)))

fig.update_xaxes(nticks=5, gridcolor='#E5E5E5', gridwidth=1)
fig.update_yaxes(showticklabels=True, linecolor='#CCCCCC')


fig.show()

### Multifamily Units Permitted Per Metro 2014-2025

In [75]:
fig = px.scatter(all_time_housing,
                 x='Roll_Year',
                 y='5 Units or More',
                 trendline_color_override="black",
                 color='Name',
                 color_discrete_sequence=px.colors.qualitative.Pastel,
                 facet_col='Name',
                 facet_col_wrap=3,
                 labels={'Roll_Year': 'Year', 'Num of Structures With 5 Units or More': 'Structures Permitted', 'Name': 'Metro Area'},
                 width=1300,
                 height=1500,)

fig.update_layout(
    title={**TITLE_STYLE, 'text': '<b>Multifamily Units Permitted (2014–2025) </b>'},
    font=dict(family='Arial', size=13),
    legend=dict(title=None),
    plot_bgcolor='white',
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.update_traces(marker=dict(size=20, symbol="square-cross", line=dict(width=2)))

fig.update_xaxes(nticks=5, gridcolor='#E5E5E5', gridwidth=1)
fig.update_yaxes(showticklabels=True, linecolor='#CCCCCC')

fig.show()

### Single Family Units Compared to Apartment Units Per Metro 2014-2025

In [76]:
fig = px.line(all_time_housing,
              x='Roll_Year',
              y=['1 Unit','5 Units or More'],
              facet_col='Name',
              facet_col_wrap=3,
              color_discrete_sequence=px.colors.qualitative.Pastel,
              markers=True,
              width=1300,
              height=1500)

fig.update_layout(
    title={**TITLE_STYLE, 'text': '<b>Single-Family vs. Apartment Unit Permits (2014-2025) </b>'},
    legend=dict(title=None)
)

fig.update_xaxes(nticks=5, title_text='Year')
fig.update_yaxes(title_text='Units Permitted')

fig.show()


### Average Housing Units per 100k people

In [77]:
average_housing_and_pop = all_years_housing_and_pop[['Name','Housing_Units_Per_100k']].groupby('Name',as_index=False).mean()
average_housing_and_pop = average_housing_and_pop.sort_values(by='Housing_Units_Per_100k')

fig = px.bar(average_housing_and_pop,
             text_auto = True,
             x='Housing_Units_Per_100k',
             y='Name',
             orientation='h',
             color_discrete_sequence=UNIT_COLORS,
             labels={'Name': 'Metro Area', 'Housing_Units_Per_100k': 'Units per 100k'}
             )

fig.update_layout(
    title={**TITLE_STYLE, 'text': '<b>Average Housing Units per 100k People (2014–2024)</b>'},
    font=dict(family='Arial', size=13),
    legend=dict(title=None),
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#E5E5E5', gridwidth=1),
    yaxis=dict(linecolor="#BDABAB")
)

fig.update_traces(insidetextfont=dict(color='white'), outsidetextfont=dict(color='#333333'))

fig.show()


### Metro's Population Change from 2014-2024

In [78]:
just_pop = all_years_housing_and_pop.query('Roll_Year == 2024 | Roll_Year == 2014')

pop_1 = just_pop [['Name','Roll_Year','Pop_Estimate']].query('Roll_Year == 2014')
pop_2 = just_pop [['Name','Roll_Year','Pop_Estimate']].query('Roll_Year == 2024')

merged_pop = pd.merge(pop_1,pop_2,"inner",on='Name')

merged_pop['Pop_Change'] = merged_pop['Pop_Estimate_y'] - merged_pop['Pop_Estimate_x']
merged_pop['Trend'] = merged_pop['Pop_Change'].apply(lambda x: 'Growth' if x >= 0 else 'Decline')

merged_pop = merged_pop.sort_values(by='Pop_Change',ascending=False)

fig = px.bar(merged_pop,
             text_auto=True,
             x='Pop_Change',
             y='Name',
             orientation='h',
             color='Trend',
             color_discrete_map={'Growth': '#7B5EA7', 'Decline': '#D94F4F'},
             labels={'Name': 'Metro Area', 'Pop_Change': 'Change Over Time'})

fig.update_layout(
    title={**TITLE_STYLE, 'text': '<b> Most Populous Metros Population Change (2014–2024)</b>'},
    font=dict(family='Arial', size=13),
    legend=dict(title=None),
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#E5E5E5', gridwidth=1),
    yaxis=dict(linecolor="#BDABAB")
)

fig.show()


In [79]:
all_years_housing_and_pop_by_year = all_years_housing_and_pop[['Name','Pop_Estimate','Roll_Year']].sort_values(by=['Name', 'Roll_Year'])

fig = px.line(all_years_housing_and_pop_by_year,
                 x='Roll_Year',
                 y='Pop_Estimate',
                 symbol='Name',
                 color='Name',
                 color_discrete_sequence=px.colors.qualitative.Pastel,
                 labels={'Name': 'Metro Area', 'Pop_Estimate': 'Population', 'Roll_Year': 'Year'},
                 width=1300,
                 height=750)

fig.update_layout(
    title={**TITLE_STYLE, 'text': '<b>Most Populous Metros Population YoY (2014–2024)</b>'},
    font=dict(family='Arial', size=13),
    legend=dict(title=None),
    xaxis=dict(gridcolor='#E5E5E5', gridwidth=1),
    yaxis=dict(linecolor='#CCCCCC'),
)

fig.update_traces(marker_size=15)
fig.update_yaxes(type='log')

fig.add_vrect(
    x0=2020,
    x1=2023,
    fillcolor="red",
    opacity=0.1,
    line_width=0,
)

fig.add_annotation(
    x=2021.5,
    y=1,
    yref='paper',
    xanchor='center',
    yanchor='top',
    text='<b>Covid-19 Pandemic</b>',
    font=dict(size=13, color='#222222', family='Arial'),
    showarrow=False,
)

fig.show()


### Rate of Housing Units per 100k people

In [ ]:
all_years_housing_and_pop_by_year = all_years_housing_and_pop[['Name','Housing_Units_Per_100k','Roll_Year']].sort_values(by=['Name', 'Roll_Year'])

fig = px.line(all_years_housing_and_pop_by_year,
                 x='Roll_Year',y='Housing_Units_Per_100k',symbol='Name',
                 color='Name',
                 color_discrete_sequence=px.colors.qualitative.Pastel,
                 labels={'Name':'Metro Area','Roll_Year': 'Year'},
                 width=2000,
                 height=750)

fig.update_layout(
    title={**TITLE_STYLE, 'text': '<b>Most Populous Metros Housing Units per 100k People</b>'}
)

fig.update_traces(marker_size=15)
fig.update_yaxes(type='log')

fig.add_vrect(
    x0=2020,
    x1=2023,
    fillcolor="red",
    opacity=0.1,
    line_width=0,
)

fig.add_annotation(
    x=2021.5,
    y=1,
    yref='paper',
    xanchor='center',
    yanchor='top',
    text='<b>Covid-19 Pandemic</b>',
    font=dict(size=13, color='#222222', family='Arial'),
    showarrow=False,
)


fig.show()


These functions make it easier to look at individual cities 

In [85]:
def extract_city(city_name:str,exclude_years:list = []) -> pd.DataFrame:
    '''Extract the specific city from the all together dataframe'''

    #Pulls all of the unique names of metro areas
    metro_areas = list(all_time_housing['Name'].unique())
    if city_name not in metro_areas:
        raise Exception('The city you entered is not in the list please try entering a city in the list')

    #Filters the df down to the specific city 
    the_city = all_time_housing.query(f"Name == '{city_name}'")

    #Drops rows that are in exclude years
    if len(exclude_years) > 0:
        the_city = the_city[~all_time_housing['Roll_Year'].isin(exclude_years)]
    
    return the_city

def extract_total_apts_permitted(city_name:str) -> int:
    '''See over a period of time how many total apartments were permitted'''

    metro_areas = list(all_time_housing['Name'].unique())
    if city_name not in metro_areas:
        raise Exception('The city you entered is not in the list please try entering a city in the list')
    
    the_city = all_time_housing.query(f"Name == '{city_name}'")

    total_units = the_city['5 Units or More'].sum()

    return total_units

A closer look at L.A Housing

In [86]:
all_la = extract_city('Los Angeles Metro')

print(all_la)

     CSA   CBSA               Name    Total   1 Unit  2 Units  3 and 4 Units  \
5    348  31080  Los Angeles Metro  26950.0   8300.0    582.0          312.0   
15   348  31080  Los Angeles Metro  34034.0   8447.0    862.0          454.0   
25   348  31080  Los Angeles Metro  32114.0   9379.0    912.0          610.0   
35   348  31080  Los Angeles Metro  31084.0  10587.0   1274.0          495.0   
45   348  31080  Los Angeles Metro  29524.0  10042.0   1528.0          522.0   
55   348  31080  Los Angeles Metro  30554.0   9306.0   1694.0          358.0   
65   348  31080  Los Angeles Metro  26930.0   9737.0   1266.0          406.0   
75   348  31080  Los Angeles Metro  31151.0  11090.0   1284.0          510.0   
85   348  31080  Los Angeles Metro  32873.0  11199.0   1626.0          414.0   
95   348  31080  Los Angeles Metro  30767.0  12035.0   1492.0          586.0   
105  348  31080  Los Angeles Metro  26781.0  11777.0   1370.0          365.0   
115  348  31080  Los Angeles Metro  2770

How to filter for specific years

In [87]:
mansion_tax_years = [2023,2024,2025]

all_la_pre_mansion_tax = all_la.query("Roll_Year != @mansion_tax_years")

all_la_post_mansion_tax = all_la.query("Roll_Year == @mansion_tax_years")

print(all_la_pre_mansion_tax)

    CSA   CBSA               Name    Total   1 Unit  2 Units  3 and 4 Units  \
5   348  31080  Los Angeles Metro  26950.0   8300.0    582.0          312.0   
15  348  31080  Los Angeles Metro  34034.0   8447.0    862.0          454.0   
25  348  31080  Los Angeles Metro  32114.0   9379.0    912.0          610.0   
35  348  31080  Los Angeles Metro  31084.0  10587.0   1274.0          495.0   
45  348  31080  Los Angeles Metro  29524.0  10042.0   1528.0          522.0   
55  348  31080  Los Angeles Metro  30554.0   9306.0   1694.0          358.0   
65  348  31080  Los Angeles Metro  26930.0   9737.0   1266.0          406.0   
75  348  31080  Los Angeles Metro  31151.0  11090.0   1284.0          510.0   
85  348  31080  Los Angeles Metro  32873.0  11199.0   1626.0          414.0   

    5 Units or More  Num of Structures With 5 Units or More  Roll_Year  \
5           17756.0                                   485.0       2014   
15          24271.0                                   550.0  